In [ ]:
!pip install -q torch transformers av pandas numpy matplotlib pillow huggingface_hub

In [ ]:
import gc
import math
import time
import warnings

import av
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from PIL import Image
from transformers import AutoModel, AutoVideoProcessor

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
MODEL_NAME = "facebook/vjepa2-vitl-fpc64-256"
DATASET_NAME = "lerobot/svla_so100_pickplace"
FRAME_STRIDE = 6
IMAGE_SIZE = 256
POOL_GRID = 8
NUM_TRAIN_EPISODES = 40
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")

# World Models

A world model predicts how an environment changes when an agent acts. With one, an agent can
plan by imagining the outcome of candidate actions instead of trying them. This notebook builds
the predictive-representation kind on real robot data: a frozen V-JEPA 2 video encoder turns
camera frames into embeddings, a small action-conditioned transformer learns to predict the next
embedding from the current one and the arm's motion, and a cross-entropy-method planner searches
for the motion that reaches a goal image. Everything is measured on held-out SO-100 pick-and-place
episodes.

---

# 1) Introduction

## 1.1 The problem

Given the current observation $o_t$ and an action $a_t$, predict the next observation, or a
representation of it:

$$
p(s_{t+1} \mid s_{\le t}, a_t)
$$

A model that answers this for any $a_t$, not only the action a demonstrator happened to take,
can be queried with actions that were never executed. That is what makes planning possible:
search over actions, imagine where each one leads, and act on the best.

## 1.2 Where earlier approaches fall short

| Approach | Limitation |
|---|---|
| Model-free RL / behaviour cloning | Maps observation to action directly; cannot evaluate an action it was not trained to output, and cannot be re-aimed at a new goal without retraining |
| Pixel prediction (video models) | Spends capacity on texture, lighting and background that do not matter for control; rendering is slow |
| Latent dynamics trained per task (World Models 2018, Dreamer) | The encoder is learned from one environment's reward and reconstruction signal; it does not transfer across scenes |
| Interactive video generators (Genie 3, Cosmos) | Photoreal and controllable, but billions of parameters and still generate pixels; not something a robot can run inside a planning loop on board |

## 1.3 The core idea

Use a general self-supervised video encoder as a frozen state estimator, and learn only the
dynamics on top of it. The encoder (V-JEPA 2) was pretrained by predicting masked video in
embedding space, so its features keep what is predictable (objects, the arm, contact) and drop
what is not (sensor noise, exact texture). The dynamics model is small and trained on a few hours
of robot video with actions. Planning is a search in embedding space:

```
 goal image ──► encoder ──► z_goal ─────────────────────────────┐
                                                                ▼
 camera o_t ──► encoder ──► z_t ──► predictor(z, a_1..a_H) ──► ẑ_{t+H} ──► energy ‖ẑ − z_goal‖₁
                                         ▲                                       │
                                         └──── CEM: sample, keep elites, refit ◄─┘
```

## 1.4 Findings from the past year

| Model | Date | What is new |
|---|---|---|
| **V-JEPA 2 / V-JEPA 2-AC** (Meta) | Jun 2025 | Encoder pretrained on 1M+ hours of video; a 300M action-conditioned predictor trained on ~62 h of DROID robot video; zero-shot pick-and-place on new lab robots by planning in embedding space |
| **Genie 3** (Google DeepMind) | Aug 2025 | Real-time interactive worlds at 720p / 24 fps from a text prompt, about a minute of visual memory; Waymo built its driving world model on it (Feb 2026) |
| **Cosmos** (NVIDIA) | 2025 | Open world foundation models (Predict / Transfer / Reason) for robot and driving synthetic data |
| **Marble** (World Labs) | 2025 | Persistent, downloadable 3D worlds from text or images |
| GR00T N2 preview | 2026 | World-model planning inside a robot foundation policy |

## 1.5 What is built here

| Piece | Built from scratch | Imported |
|---|---|---|
| Frame encoder | Frame-pair input, token pooling | V-JEPA 2 ViT-L weights (`transformers`) |
| Action-conditioned predictor | Token layout, residual head, teacher-forced and rollout losses | `nn.TransformerEncoderLayer` |
| Baselines | Copy-last, action-free predictor, shuffled actions | none |
| Planner | Cross-entropy method over action sequences, $L_1$ energy | none |
| Evaluation | Rollout error by horizon, planned vs true arm motion | none |

---

# 2) Setup

## 2.1 Data: SO-100 pick-and-place

`lerobot/svla_so100_pickplace` is 50 teleoperated episodes of a low-cost 6-joint SO-100 arm
picking up a cube and placing it in a box, recorded at 30 fps from a top camera and a wrist
camera, with the joint state and the commanded joint targets at every frame. The episodes are
stored as one AV1 video per camera; row $i$ of the parquet table is frame $i$ of that video.

V-JEPA 2-AC plans at 4 fps, because consecutive 30 fps frames barely differ. Keeping every 6th
frame gives 5 Hz, about 0.2 s of motion per step. Episodes 0–39 train the predictor, 40–49 are
held out for every number below.

In [ ]:
root = snapshot_download(DATASET_NAME, repo_type="dataset")
table = pd.read_parquet(f"{root}/data/chunk-000/file-000.parquet")
table = table[table["frame_index"] % FRAME_STRIDE == 0].reset_index(drop=True)
keep_rows = set(table["index"].tolist())

start = time.time()
frames = []
with av.open(f"{root}/videos/observation.images.top/chunk-000/file-000.mp4") as video:
    for i, frame in enumerate(video.decode(video=0)):
        if i in keep_rows:
            image = frame.to_image().resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
            frames.append(np.asarray(image))
frames = np.stack(frames)
episode = table["episode_index"].to_numpy()
state = torch.tensor(np.stack(table["observation.state"].to_numpy()))
print(
    f"frames {tuple(frames.shape)} | state {tuple(state.shape)} | "
    f"{time.time() - start:.0f} s"
)
print(
    f"{len(np.unique(episode))} episodes, {np.bincount(episode).mean():.0f} steps each"
)

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
first = np.where(episode == NUM_TRAIN_EPISODES)[0]
for ax, i in zip(axes, np.linspace(0, len(first) - 1, 6).astype(int)):
    ax.imshow(frames[first[i]])
    ax.set_title(f"step {i} ({i / 5:.1f} s)")
    ax.axis("off")
plt.suptitle(f"held-out episode {NUM_TRAIN_EPISODES}, top camera at 5 Hz")
plt.show()

## 2.2 Encoder: frozen V-JEPA 2 ViT-L

V-JEPA 2 is a video ViT whose patch embedding is a **tubelet** of 2 frames × 16 × 16 pixels. A
single image is encoded by repeating it to a 2-frame clip, exactly how V-JEPA 2-AC encodes each
robot frame. At 256 × 256 this gives a 16 × 16 grid of 1024-d tokens:

```
 frame (256, 256, 3) ──repeat──► clip (2, 3, 256, 256) ──ViT-L──► (256 tokens, 1024)
                      ──layer norm, 2×2 average pool──► z_t (64 tokens, 1024)
```

The predictor works on the 8 × 8 pooled grid. A 4 × 4 grid (or a single pooled vector) averages
the small arm into the background, and a linear probe from the change in embedding to the joint
motion gets clearly weaker. The encoder runs once over all frames
and is then freed; only the 64 × 1024 embeddings are kept.

In [ ]:
def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


def free_device_memory() -> None:
    """Release cached accelerator memory after a model is deleted."""
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()


processor = AutoVideoProcessor.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME, dtype=torch.float16).to(DEVICE).eval()
num_params = sum(p.numel() for p in encoder.parameters())
print(
    f"V-JEPA 2 encoder: {num_params / 1e6:.0f} M params | "
    f"memory {device_memory_gb():.2f} GB"
)

In [ ]:
@torch.no_grad()
def encode_frames(images: np.ndarray, chunk: int = 32) -> torch.Tensor:
    """Embed frames (num_frames, H, W, 3) -> (num_frames, POOL_GRID**2, embed_dim)."""
    out = []
    for b in range(0, len(images), chunk):
        # each frame becomes a 2-frame clip: (chunk, 2, H, W, 3)
        clips = np.repeat(images[b : b + chunk, None], 2, axis=1)
        pixel_values = processor(
            list(clips), do_resize=False, do_center_crop=False, return_tensors="pt"
        )["pixel_values_videos"].to(DEVICE, torch.float16)
        hidden = encoder(pixel_values_videos=pixel_values, skip_predictor=True)
        hidden = hidden.last_hidden_state.float()
        embed_dim = hidden.shape[-1]
        hidden = F.layer_norm(hidden, (embed_dim,))
        # (chunk, 256, embed_dim) -> (chunk, embed_dim, 16, 16) -> pooled grid
        side = int(math.isqrt(hidden.shape[1]))
        grid = hidden.view(-1, side, side, embed_dim).permute(0, 3, 1, 2)
        grid = F.adaptive_avg_pool2d(grid, POOL_GRID)
        out.append(grid.flatten(2).transpose(1, 2))
    return torch.cat(out)


print(encode_frames(frames[:2]).shape)

In [ ]:
start = time.time()
z = encode_frames(frames)
print(
    f"embeddings {tuple(z.shape)} | {time.time() - start:.0f} s "
    f"({len(frames) / (time.time() - start):.0f} frames/s) | "
    f"peak memory {device_memory_gb():.2f} GB"
)
del encoder
free_device_memory()

---

# 3) The math

## 3.1 What the encoder was trained for

V-JEPA 2 was pretrained without labels by masking spatio-temporal blocks of a video $x$ and
predicting the *embeddings* of the masked part from the visible part:

$$
\mathcal{L}_{\text{JEPA}} = \big\lVert P_\psi\big(f_\theta(x_{\text{ctx}}), \Delta\big)
- \operatorname{sg}\big(f_{\bar\theta}(x_{\text{tgt}})\big) \big\rVert_1
$$

$f_\theta$ is the encoder, $f_{\bar\theta}$ its exponential moving average (the target encoder),
$\Delta$ the positions of the masked tokens, $P_\psi$ a predictor, and $\operatorname{sg}$ stops
the gradient. Because the target is an embedding rather than pixels, the encoder is free to drop
whatever cannot be predicted from context. Only $f_\theta$ is used here, frozen.

## 3.2 Action-conditioned prediction

Let $z_t \in \mathbb{R}^{N \times D}$ be the $N = 64$ pooled tokens ($D = 1024$) of frame $t$,
$s_t \in \mathbb{R}^6$ the joint state and $a_t = s_{t+1} - s_t$ the joint motion over the next
step (V-JEPA 2-AC uses the end-effector delta; SO-100 records joint space only). Both are
standardised per joint with training-set statistics. The predictor $P_\phi$ outputs a residual
on the current embedding:

$$
\hat z_{t+1} = z_t + P_\phi(z_t, a_t, s_t)
$$

so the copy-last baseline $\hat z_{t+1} = z_t$ is the predictor at initialisation. Two losses,
as in V-JEPA 2-AC:

$$
\mathcal{L}_{\text{tf}} = \big\lVert \hat z_{t+1} - z_{t+1} \big\rVert_1, \qquad
\mathcal{L}_{\text{roll}} = \big\lVert \hat z_{t+2} - z_{t+2} \big\rVert_1, \quad
\hat z_{t+2} = \hat z_{t+1} + P_\phi(\hat z_{t+1}, a_{t+1}, s_{t+1})
$$

$\mathcal{L}_{\text{tf}}$ (teacher forcing) feeds the real $z_t$. $\mathcal{L}_{\text{roll}}$
feeds the model its own prediction, which is what happens at planning time; without it,
multi-step errors compound because the model never saw its own outputs as input.
$\lVert\cdot\rVert_1$ is the mean absolute error over tokens and channels.

## 3.3 Planning by energy minimisation

Given the current frame and a goal image with embedding $z_{\text{goal}}$, roll the predictor
out for $H$ steps and score the action sequence by how close it lands:

$$
E(a_{1:H}) = \big\lVert \hat z_{t+H}(z_t, s_t, a_{1:H}) - z_{\text{goal}} \big\rVert_1,
\qquad s_{t+h+1} = s_{t+h} + a_{t+h}
$$

$E$ is not differentiable through anything a robot would trust, and it has many local minima,
so V-JEPA 2-AC minimises it with the **cross-entropy method** (CEM). Keep a diagonal Gaussian
over the $H \times 6$ action sequence, initialised at $\mu = 0, \sigma = 1$ (one standard
deviation of real motion per step). For $I$ iterations: draw $M$ samples, evaluate $E$ for all
of them in one batched rollout, keep the $K$ lowest (the elites), and refit $\mu, \sigma$ to the
elites. In model-predictive control the robot executes $a_1$, observes, and replans.

There is no robot here, so planning is scored offline: the goal is the real frame $H$ steps
later in a held-out episode, and the planned total motion $\sum_h \mu_h$ is compared with the
motion the teleoperator actually made, $s_{t+H} - s_t$.

| Symbol | Meaning | Value here |
|---|---|---|
| $N, D$ | Tokens per frame, embedding width | 64, 1024 |
| $H$ | Planning horizon (5 Hz steps) | 3 |
| $M, K, I$ | CEM samples, elites, iterations | 256, 25, 6 |

---

# 4) From-scratch implementation

## 4.1 Actions, normalisation and splits

A training pair is a step $t \to t+1$ inside one episode. `has_next[t]` marks steps whose next
frame is in the same episode; the last step of each episode has none.

In [ ]:
has_next = np.r_[episode[1:] == episode[:-1], False]
action = torch.zeros_like(state)
action[:-1] = state[1:] - state[:-1]
is_train = episode < NUM_TRAIN_EPISODES

state_mean = state[is_train].mean(0)
state_std = state[is_train].std(0)
action_std = action[is_train & has_next].std(0)
state_norm = ((state - state_mean) / state_std).to(DEVICE)
action_norm = (action / action_std).to(DEVICE)
# one unit of normalised action expressed in normalised state units
ACTION_TO_STATE = (action_std / state_std).to(DEVICE)
train_steps = np.where(is_train & has_next)[0]
test_steps = np.where(~is_train & has_next)[0]
z = z.to(DEVICE)

joints = ["pan", "lift", "elbow", "wrist flex", "wrist roll", "gripper"]
print(f"train steps {len(train_steps)} | test steps {len(test_steps)}")
print(
    "joint motion std per 0.2 s (deg):",
    dict(zip(joints, action_std.round(decimals=1).tolist())),
)

In [ ]:
def steps_with_horizon(steps: np.ndarray, horizon: int) -> np.ndarray:
    """Keep steps t whose frame t + horizon is in the same episode."""
    ok = steps + horizon < len(episode)
    steps = steps[ok]
    return steps[episode[steps + horizon] == episode[steps]]


def copy_error(steps: np.ndarray, horizon: int) -> float:
    """L1 between z_t and z_{t+horizon}: the error of predicting no change."""
    return F.l1_loss(z[steps], z[steps + horizon]).item()


for horizon in [1, 2, 4]:
    print(
        f"copy-last L1 at {horizon} step(s): "
        f"{copy_error(steps_with_horizon(test_steps, horizon), horizon):.4f}"
    )

## 4.2 The predictor

Each frame's 64 tokens are projected to a narrow width, one action token and one state token are
appended, and a 4-layer transformer mixes them; the 64 outputs are projected back to 1024 and
added to $z_t$. The output projection starts at zero so training starts from the copy baseline.

```
 z_t (B, 64, 1024) ──Linear──► (B, 64, 256) ┐
 a_t (B, 6) ──Linear──► (B, 1, 256)          ├─ concat + pos ─► 4 × TransformerEncoderLayer
 s_t (B, 6) ──Linear──► (B, 1, 256)          ┘        │
                                       first 64 tokens ─► Linear(256→1024) ─► + z_t ─► ẑ_{t+1}
```

V-JEPA 2-AC's predictor is the same idea at 300M parameters, with a block-causal mask over a
history of frames; one frame of context plus the joint state keeps this one at a few million.

In [ ]:
class ActionConditionedPredictor(nn.Module):
    """Residual next-embedding predictor P(z_t, a_t, s_t) -> z_{t+1}."""

    def __init__(
        self,
        num_tokens: int,
        embed_dim: int,
        hidden_dim: int = 256,
        num_heads: int = 4,
        num_layers: int = 4,
        action_dim: int = 6,
    ) -> None:
        super().__init__()
        self.num_tokens = num_tokens
        self.frame_in = nn.Linear(embed_dim, hidden_dim)
        self.action_in = nn.Linear(action_dim, hidden_dim)
        self.state_in = nn.Linear(action_dim, hidden_dim)
        self.pos = nn.Parameter(0.02 * torch.randn(num_tokens + 2, hidden_dim))
        layer = nn.TransformerEncoderLayer(
            hidden_dim,
            num_heads,
            4 * hidden_dim,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(
            layer, num_layers, enable_nested_tensor=False
        )
        self.frame_out = nn.Linear(hidden_dim, embed_dim)
        nn.init.zeros_(self.frame_out.weight)
        nn.init.zeros_(self.frame_out.bias)

    def forward(
        self,
        z_t: torch.Tensor,  # (batch_num, num_tokens, embed_dim)
        a_t: torch.Tensor,  # (batch_num, action_dim)
        s_t: torch.Tensor,  # (batch_num, action_dim)
    ) -> torch.Tensor:
        """Return the predicted next embedding (batch_num, num_tokens, embed_dim)."""
        tokens = torch.cat(
            [
                self.frame_in(z_t),
                self.action_in(a_t)[:, None],
                self.state_in(s_t)[:, None],
            ],
            dim=1,
        )
        hidden = self.blocks(tokens + self.pos)
        return z_t + self.frame_out(hidden[:, : self.num_tokens])


predictor = ActionConditionedPredictor(z.shape[1], z.shape[2]).to(DEVICE)
batch = torch.tensor(train_steps[:4], device=DEVICE)
pred = predictor(z[batch], action_norm[batch], state_norm[batch])
print(
    f"{sum(p.numel() for p in predictor.parameters()) / 1e6:.1f} M params | "
    f"out {tuple(pred.shape)}"
)
print("equals copy baseline at init:", torch.equal(pred, z[batch]))

## 4.3 Losses

Both losses share one sampled batch of steps $t$ that have frames $t+1$ and $t+2$ in the same
episode. `action_gain` is 1 for the action-conditioned model and 0 for the **action-free**
control, which sees the same frames and joint states but never the motion. That control answers
the question the copy baseline cannot: does the model use $a_t$, or does it only learn the
average next frame of a demonstration?

In [ ]:
def predictor_losses(
    model: ActionConditionedPredictor, steps: torch.Tensor, action_gain: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Teacher-forced and 2-step rollout L1 losses for a batch of start steps t."""
    a_0 = action_norm[steps] * action_gain
    a_1 = action_norm[steps + 1] * action_gain
    z_1 = model(z[steps], a_0, state_norm[steps])
    z_2 = model(z_1, a_1, state_norm[steps + 1])
    return F.l1_loss(z_1, z[steps + 1]), F.l1_loss(z_2, z[steps + 2])


rollout_train_steps = steps_with_horizon(train_steps, 2)
loss_tf, loss_roll = predictor_losses(predictor, batch, 1.0)
print(f"at init: teacher-forced {loss_tf.item():.4f} | rollout {loss_roll.item():.4f}")

## 4.4 Multi-step rollout

The same function rolls out true actions (to measure prediction error), shuffled actions (to
check the model depends on them) and CEM candidates (to plan). The joint state is advanced with
the action, $s_{t+h+1} = s_{t+h} + a_{t+h}$, so a candidate that moves the arm also changes the
state token the next step sees.

In [ ]:
@torch.no_grad()
def rollout(
    model: ActionConditionedPredictor,
    z_start: torch.Tensor,  # (batch_num, num_tokens, embed_dim)
    s_start: torch.Tensor,  # (batch_num, 6)
    actions: torch.Tensor,  # (batch_num, horizon, 6)
) -> torch.Tensor:
    """Embedding after applying actions: -> (batch_num, num_tokens, embed_dim)."""
    z_pred, s_pred = z_start, s_start
    for h in range(actions.shape[1]):
        z_pred = model(z_pred, actions[:, h], s_pred)
        s_pred = s_pred + actions[:, h] * ACTION_TO_STATE
    return z_pred


def true_actions(steps: np.ndarray, horizon: int) -> torch.Tensor:
    """Recorded normalised actions a_t..a_{t+horizon-1}: -> (len(steps), horizon, 6)."""
    return torch.stack([action_norm[steps + h] for h in range(horizon)], dim=1)


s = steps_with_horizon(test_steps, 4)[:3]
print(rollout(predictor, z[s], state_norm[s], true_actions(s, 4)).shape)

## 4.5 The CEM planner

All candidate sequences for a chunk of start states are rolled out as one batch of
`chunk × num_samples` sequences. The elite standard deviation gets a small floor so the search
never collapses to a point before the last iteration.

In [ ]:
@torch.no_grad()
def cem_plan(
    model: ActionConditionedPredictor,
    steps: np.ndarray,
    horizon: int,
    num_samples: int = 256,
    num_elites: int = 25,
    num_iters: int = 6,
    chunk: int = 4,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Plan z_t -> z_{t+horizon}: -> plans (len(steps), horizon, 6), energies."""
    plans, energies = [], []
    for b in range(0, len(steps), chunk):
        part = steps[b : b + chunk]
        num = len(part)
        z_start = z[part].repeat_interleave(num_samples, 0)
        s_start = state_norm[part].repeat_interleave(num_samples, 0)
        goal = z[part + horizon][:, None]
        mean = torch.zeros(num, horizon, 6, device=DEVICE)
        std = torch.ones(num, horizon, 6, device=DEVICE)
        for _ in range(num_iters):
            # (num, num_samples, horizon, 6)
            samples = mean[:, None] + std[:, None] * torch.randn(
                num, num_samples, horizon, 6, device=DEVICE
            )
            z_end = rollout(model, z_start, s_start, samples.flatten(0, 1))
            energy = (
                (z_end.view(num, num_samples, *z_end.shape[1:]) - goal)
                .abs()
                .mean((2, 3))
            )
            elite_idx = energy.topk(num_elites, dim=1, largest=False).indices
            elites = torch.gather(
                samples, 1, elite_idx[:, :, None, None].expand(-1, -1, horizon, 6)
            )
            mean, std = elites.mean(1), elites.std(1) + 0.05
        plans.append(mean)
        final = rollout(model, z[part], state_norm[part], mean)
        energies.append((final - goal[:, 0]).abs().mean((1, 2)))
    return torch.cat(plans), torch.cat(energies)


plan, energy = cem_plan(
    predictor, steps_with_horizon(test_steps, 3)[:2], 3, num_iters=2
)
print(f"plan {tuple(plan.shape)} | energy {energy.cpu().numpy().round(4)}")

---

# 5) End-to-end run

## 5.1 Training

Two predictors with identical architecture, data and steps: one sees the actions, one does not.
1,000 AdamW steps of 64 pairs each is about 25 passes over the 40 training episodes, a few
minutes per model. V-JEPA 2-AC trains its 300M predictor on 62 hours of video, about 600 times
this data.

In [ ]:
def train_predictor(
    action_gain: float, num_steps: int = 1000, batch_num: int = 64
) -> tuple[ActionConditionedPredictor, list[float]]:
    """Train a predictor on the training episodes; return it and its loss history."""
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    model = ActionConditionedPredictor(z.shape[1], z.shape[2]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
    history = []
    start = time.time()
    for step in range(num_steps):
        steps = torch.tensor(rng.choice(rollout_train_steps, batch_num), device=DEVICE)
        loss_tf, loss_roll = predictor_losses(model, steps, action_gain)
        loss = loss_tf + loss_roll
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
        if step % 250 == 0 or step == num_steps - 1:
            print(
                f"  step {step:4d} | tf {loss_tf.item():.4f} | "
                f"roll {loss_roll.item():.4f}"
            )
    print(f"  {time.time() - start:.0f} s | memory {device_memory_gb():.2f} GB")
    return model.eval(), history


print("action-conditioned")
model_ac, history_ac = train_predictor(1.0)
print("action-free")
model_free, history_free = train_predictor(0.0)

In [ ]:
window = 50
for history, label in [
    (history_ac, "action-conditioned"),
    (history_free, "action-free"),
]:
    smooth = np.convolve(history, np.ones(window) / window, mode="valid")
    plt.plot(smooth, label=label)
plt.xlabel("step")
plt.ylabel("teacher-forced + rollout L1 (train)")
plt.legend()
plt.show()

## 5.2 Prediction error on held-out episodes

For every held-out step, roll each model out $H$ steps with the recorded actions and measure
$\lVert \hat z_{t+H} - z_{t+H} \rVert_1$. The **shuffled** row feeds the action-conditioned model
action sequences taken from other, random test steps: if it relied on the actions, wrong ones
should hurt.

In [ ]:
def rollout_error(
    model: ActionConditionedPredictor,
    steps: np.ndarray,
    actions: torch.Tensor,
    horizon: int,
) -> float:
    """Mean L1 between the rolled-out and the real embedding at t + horizon."""
    z_end = rollout(model, z[steps], state_norm[steps], actions)
    return F.l1_loss(z_end, z[steps + horizon]).item()


generator = torch.Generator().manual_seed(SEED)
rows = []
for horizon in [1, 2, 4]:
    steps = steps_with_horizon(test_steps, horizon)
    actions = true_actions(steps, horizon)
    shuffled = actions[torch.randperm(len(steps), generator=generator).to(DEVICE)]
    rows.append(
        {
            "horizon (s)": horizon / 5,
            "copy-last": copy_error(steps, horizon),
            "action-free": rollout_error(model_free, steps, 0 * actions, horizon),
            "action-cond., shuffled a": rollout_error(
                model_ac, steps, shuffled, horizon
            ),
            "action-cond.": rollout_error(model_ac, steps, actions, horizon),
        }
    )
print(pd.DataFrame(rows).round(4).to_string(index=False))

Both predictors beat copy-last by a wide margin; most of that is learning how a demonstration
typically continues from the current frame and joint state, which the action-free model can do
too. At 0.2 s the two are within 0.001. The gap opens with the horizon (0.231 vs 0.222 at
0.8 s), and the shuffled row is the clearer evidence: wrong actions make the action-conditioned
model worse than having no actions at all, so it is steering its prediction by $a_t$.

## 5.3 Looking at a prediction

Embeddings cannot be viewed directly, and this model has no pixel decoder by design. One way to
look at a prediction is retrieval: find the real frame of the same episode whose embedding is
nearest to $\hat z_{t+H}$. If the prediction is right, the retrieved frame is $t+H$ or a
neighbour. The mean distance in steps between the retrieved frame and $t+H$ summarises this over
all held-out steps; copy-last always retrieves frame $t$, an error of $H$.

In [ ]:
def retrieved_offset(
    z_pred: torch.Tensor, steps: np.ndarray, horizon: int
) -> np.ndarray:
    """Step index of each prediction's nearest same-episode frame, minus t + horizon."""
    offsets = np.zeros(len(steps), dtype=int)
    for e in np.unique(episode[steps]):
        rows = np.where(episode[steps] == e)[0]
        frame_ids = np.where(episode == e)[0]
        dist = torch.cdist(z_pred[rows].flatten(1), z[frame_ids].flatten(1))
        nearest = frame_ids[dist.argmin(1).cpu().numpy()]
        offsets[rows] = nearest - (steps[rows] + horizon)
    return offsets


HORIZON_VIEW = 4
steps = steps_with_horizon(test_steps, HORIZON_VIEW)
actions = true_actions(steps, HORIZON_VIEW)
z_pred_ac = rollout(model_ac, z[steps], state_norm[steps], actions)
z_pred_free = rollout(model_free, z[steps], state_norm[steps], 0 * actions)
off_ac = retrieved_offset(z_pred_ac, steps, HORIZON_VIEW)
off_free = retrieved_offset(z_pred_free, steps, HORIZON_VIEW)
print(
    f"mean |retrieved - (t+{HORIZON_VIEW})| in steps: copy {HORIZON_VIEW} | "
    f"action-free {np.abs(off_free).mean():.2f} | "
    f"action-cond. {np.abs(off_ac).mean():.2f}"
)

In [ ]:
# the held-out step where the arm moves most over the horizon
motion = (state[steps + HORIZON_VIEW] - state[steps]).abs().sum(1).numpy()
k = int(motion.argmax())
t = steps[k]
panels = [
    (t, "now: t"),
    (t + HORIZON_VIEW, f"real t+{HORIZON_VIEW}"),
    (t + HORIZON_VIEW + off_ac[k], f"nearest to action-cond. ({off_ac[k]:+d})"),
    (t + HORIZON_VIEW + off_free[k], f"nearest to action-free ({off_free[k]:+d})"),
]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, (i, title) in zip(axes, panels):
    ax.imshow(frames[i])
    ax.set_title(title)
    ax.axis("off")
plt.suptitle(f"episode {episode[t]}, {HORIZON_VIEW / 5:.1f} s ahead")
plt.show()

The improvement is real but small: predictions trained with $L_1$ regress toward the
conditional mean, an embedding that blends the possible futures and so sits close to the
present. The action-free model, which averages over more futures, does no better than copying.

## 5.4 Planning toward a goal image

The planning test takes 60 held-out steps where the arm moves (above-median motion over the
horizon; for a stationary arm "plan zero motion" is trivially right), uses the real frame
$H = 3$ steps (0.6 s) later as the goal, and lets CEM find the action sequence. Baselines:
**zero** motion and **random** motion (a real action sequence from a random training step).
The planner never sees the true actions.

In [ ]:
def planning_set(horizon: int, num: int = 60) -> np.ndarray:
    """Held-out steps with above-median arm motion over the horizon."""
    steps = steps_with_horizon(test_steps, horizon)
    motion = true_actions(steps, horizon).sum(1).norm(dim=1).cpu().numpy()
    moving = steps[motion > np.median(motion)]
    return np.random.default_rng(SEED).choice(moving, num, replace=False)


def score_plan(plan: torch.Tensor, steps: np.ndarray, horizon: int) -> dict[str, float]:
    """Compare planned total motion with the recorded one (normalised units)."""
    true_total = true_actions(steps, horizon).sum(1).cpu()
    plan_total = plan.sum(1).cpu()
    rng = np.random.default_rng(SEED)
    random_total = true_actions(
        rng.choice(steps_with_horizon(train_steps, horizon), len(steps)), horizon
    )
    random_total = random_total.sum(1).cpu()
    corr = [np.corrcoef(plan_total[:, j], true_total[:, j])[0, 1] for j in range(6)]
    return {
        "MAE zero": true_total.abs().mean().item(),
        "MAE random": (random_total - true_total).abs().mean().item(),
        "MAE CEM": (plan_total - true_total).abs().mean().item(),
        **{f"corr {name}": c for name, c in zip(joints, corr)},
    }


HORIZON_PLAN = 3
plan_steps = planning_set(HORIZON_PLAN)
start = time.time()
plan, plan_energy = cem_plan(model_ac, plan_steps, HORIZON_PLAN)
plan_seconds = time.time() - start
print(
    f"CEM on {len(plan_steps)} goals: {plan_seconds:.0f} s | "
    f"memory {device_memory_gb():.2f} GB"
)
scores = score_plan(plan, plan_steps, HORIZON_PLAN)
print(pd.Series(scores).round(3).to_string())

Is the planner or the model the limit? Compare the energy of the CEM plan with the energy of the
recorded actions under the same model. If CEM finds lower energy than the truth, the search is
doing its job and the remaining error comes from the model's view of the world.

In [ ]:
true_end = rollout(
    model_ac,
    z[plan_steps],
    state_norm[plan_steps],
    true_actions(plan_steps, HORIZON_PLAN),
)
true_energy = (true_end - z[plan_steps + HORIZON_PLAN]).abs().mean((1, 2))
print(
    f"energy: CEM plan {plan_energy.mean().item():.4f} | "
    f"recorded actions {true_energy.mean().item():.4f}"
)
print(
    f"CEM plan below recorded actions on "
    f"{(plan_energy < true_energy).float().mean().item():.0%} of goals"
)

true_total = true_actions(plan_steps, HORIZON_PLAN).sum(1).cpu() * action_std
plan_total = plan.sum(1).cpu() * action_std
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, j in zip(axes, [0, 1]):
    ax.scatter(true_total[:, j], plan_total[:, j], s=12)
    lim = true_total[:, j].abs().max().item()
    ax.plot([-lim, lim], [-lim, lim], "k--", lw=1)
    ax.set_xlabel(f"recorded {joints[j]} motion over 0.6 s (deg)")
    ax.set_ylabel("planned (deg)")
plt.tight_layout()
plt.show()

---

# 6) Variants

## 6.1 Planning horizon and number of samples

A longer horizon puts the goal further away, so the arm's motion is a larger part of the change
in embedding; it also makes the rollout longer and the search space bigger ($6H$ dimensions).
Fewer CEM samples make planning cheaper, which matters on a robot that replans every step.

In [ ]:
def variant_row(
    horizon: int, num_samples: int, seconds: float, scores: dict[str, float]
) -> dict[str, float]:
    """One row of the horizon / sample-count table."""
    keep = ["MAE zero", "MAE CEM", "corr pan", "corr lift"]
    row = {"H": horizon, "samples": num_samples, "s / goal": seconds / 60}
    return row | {key: scores[key] for key in keep}


rows = []
for horizon, num_samples in [(1, 256), (2, 256), (3, 32)]:
    steps = planning_set(horizon)
    start = time.time()
    plan_h, _ = cem_plan(
        model_ac,
        steps,
        horizon,
        num_samples=num_samples,
        num_elites=max(4, num_samples // 10),
    )
    seconds = time.time() - start
    rows.append(
        variant_row(horizon, num_samples, seconds, score_plan(plan_h, steps, horizon))
    )
rows.insert(2, variant_row(HORIZON_PLAN, 256, plan_seconds, scores))
print(pd.DataFrame(rows).round(3).to_string(index=False))

---

# 7) Practical notes

## 7.1 What the run showed

* **Planning works on the joints the camera sees well.** At $H = 3$ the planned shoulder-pan
  motion correlates about 0.95 with what the operator did, shoulder lift about 0.85; wrist roll
  and gripper, a few pixels in a top-down view, are weakest. CEM's total error is about 70%
  of predicting no motion.
* **The model, not the search, is the bottleneck.** CEM reaches lower energy than the recorded
  actions on every goal: by the model's own measure it finds a better route than the truth, so
  better planning needs a better predictor or a sharper energy, not more samples.
* **Very short horizons do not plan well.** At $H = 1$ (0.2 s) the motion is small relative to
  embedding noise, and CEM's MAE is slightly worse than predicting zero even though the
  direction is right (pan correlation about 0.6). Longer horizons make the goal informative.
* **Samples are cheap to cut.** 32 samples instead of 256 is about eight times faster and loses
  little on the large joints.

## 7.2 Notes

| Topic | Note |
|---|---|
| Token resolution | Pool the encoder grid only as far as the controlled object survives; the arm is a few patches of a 16 × 16 grid |
| Action space | Joint deltas here; V-JEPA 2-AC uses end-effector deltas, which transfer across robots with different kinematics |
| Demonstration bias | Teleoperated data contains only the actions a competent operator took, so the action-free model is a strong baseline on prediction error; counterfactual tests (shuffled actions, planning) are what show the actions are used |
| Planning cost | Every CEM iteration is `samples × horizon` predictor calls; V-JEPA 2-AC needs about 16 s per action on a GPU at 300M params |
| Energy landscape | $L_1$ to a goal embedding is flat while the gripper is far from the object and becomes sharp near contact; long tasks need sub-goal images |
| Camera | V-JEPA 2-AC fails when the camera moves between training and deployment; the embedding encodes the viewpoint as much as the scene |
| Production | Larger predictor with a block-causal history, more robot hours (DROID-scale), closed-loop MPC on the robot, a gripper-aware energy term |

---

# 8) Conclusion

| Question | Answer from this run |
|---|---|
| Can a frozen self-supervised video encoder serve as the state for a world model? | Yes: with 64 pooled V-JEPA 2 tokens, a 3.7M-parameter predictor trained on 40 episodes learns dynamics that beat copy-last at every horizon |
| Does the predictor use the actions? | Yes, but demonstrations hide it: the action-free control is close on prediction error, and shuffled actions are what expose the dependence |
| Can it plan? | Toward a goal 0.6 s ahead, CEM recovers the operator's large-joint motion (pan correlation ~0.95) without ever seeing actions at test time |
| What limits it? | The predictor: CEM already beats the true actions on the model's energy |
| What production adds | A 300M block-causal predictor, tens of hours of robot video, end-effector actions, closed-loop MPC on hardware |

The recipe has three pieces: a general encoder that is never fine-tuned, a small dynamics model
trained on a little robot data, and a sampling planner over its rollouts. It runs without pixel
generation, and a new goal is just a new image.